In [1]:
import streamlit as st
import pandas as pd
import time

import requests
from bs4 import BeautifulSoup
from webdriver_manager.chrome import ChromeDriverManager

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By

In [3]:
# Selenium으로 HTML 가져오기
service = Service(executable_path=ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

url = 'https://www.jobkorea.co.kr/'
driver.get(url)

wait = WebDriverWait(driver, 10)

In [4]:
# 데이터분석 입력
search_box = driver.find_element(By.CSS_SELECTOR, '#stext')
search_box.send_keys('데이터분석') 
time.sleep(2)
print('검색어 입력 완료') 

# # 검색 버튼 클릭
search_button = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, '//*[@id="common_search_btn"]')))
search_button.click()
time.sleep(2)
print("검색 버튼을 클릭했습니다.")

검색어 입력 완료
검색 버튼을 클릭했습니다.


In [5]:
# 채용 정보 추출
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')
items = soup.find_all('div', class_='Flex_display_flex__i0l0hl2 Flex_gap_space28__i0l0hl2a styles_p_space28__dk46ts8d')

jobkorea_data = []

for item in items:  
    try:
        # 회사 추출
        corp_element = item.find('div', class_='Flex_display_flex__i0l0hl2 Flex_align_center__i0l0hl8 Flex_justify_space-between__i0l0hlf styles_mb_space4__dk46ts23')
        company = corp_element.get_text().strip() if corp_element else ''

        # 제목 추출
        title_element = item.find('span', class_='Typography_variant_size18__344nw25 Typography_weight_medium__344nw2d Typography_color_gray900__344nw2l')
        title = title_element.get_text().strip() if title_element else ''

        # detail 추출
        condition_element = item.find('div', class_='Flex_display_flex__i0l0hl2 Flex_gap_space16__i0l0hlj Flex_direction_row__i0l0hl3')
        conditions = condition_element.find_all('span')
        detail = []
        for condition in conditions:
            detail.append(condition.get_text().strip())
            
        # url 추출
        link_element = item.find('a', class_='sn28bt0')
        url = link_element.get('href') if link_element else ''

        # 데이터 저장
        jobkorea_data.append({
            'Site': 'Job_Korea',
            'Col_Company': company,
            'Col_Recuit': title,
            'Col_detail': detail,
            'Col_url': url
        })
        
    except Exception as e:
        print(f"데이터 추출 중 오류 발생: {e}")
        continue

# 종료
driver.quit()

# 데이터프레임 생성
jobkorea_df = pd.DataFrame(jobkorea_data)

# 데이터프레임 출력
jobkorea_df.head()

,Site,Col_Company,Col_Recuit,Col_detail,Col_url
0,Job_Korea,㈜쿡섬,"데이터분석전문가,Python,10월부터","[경력9년↑, 학력무관, 계약직 외 1, 서울 중구]",https://www.jobkorea.co.kr/Recruit/GI_Read/476...
1,Job_Korea,㈜원익피앤이,[원익PNE] 데이터분석 및 솔루션 담당자,"[경력3년↑, 석사↑, 정규직, 경기 수원시]",https://www.jobkorea.co.kr/Recruit/GI_Read/476...
2,Job_Korea,콘센트릭스서비스코리아,[Catalyst] 데이터분석 컨설턴트,"[경력5년↑, 학력무관, 정규직, 서울 강남구]",https://www.jobkorea.co.kr/Recruit/GI_Read/475...
3,Job_Korea,㈜지바이크,[지쿠] 전략마케팅팀 데이터분석 전문가 경력 채용,"[경력3년↑, 대졸↑, 정규직, 서울 강남구]",https://www.jobkorea.co.kr/Recruit/GI_Read/474...
4,Job_Korea,넛지헬스케어㈜,[캐시워크-병역특례] 데이터분석 담당 산업기능요원,"[경력무관, 학력무관, 병역특례, 서울 강남구]",https://www.jobkorea.co.kr/Recruit/GI_Read/474...


In [6]:
# CSV 파일로 저장
jobkorea_df.to_csv('data_temp/data_jobkorea.csv', index=False, encoding='utf-8-sig')
print(f"총 {len(jobkorea_df)}개의 채용공고가 저장되었습니다.")

총 20개의 채용공고가 저장되었습니다.
